# 📊 Dashboard Prototype - Live Editor

**Purpose:** Live prototyping of stakeholder dashboard columns and formatting without downloading Excel files.

## Instructions
1. Run all cells to load the dashboard data
2. Edit the `edited_df` DataFrame in subsequent cells
3. Re-run cells to see changes reflected in the table
4. Use the column configuration to test different formats

In [ ]:
# Load required libraries
import pandas as pd
import numpy as np
from IPython.display import display, HTML

# File paths
ETF_CSV = "../data/outputs/etf_prices.csv"
WEO_CSV = "../data/outputs/weo_gdp.csv"
METADATA_CSV = "../data/outputs/etf_ticker_metadata.csv"
CRUDE_IMPACT_CSV = "../data/outputs/crude_oil_import_impact.csv"
REER_CSV = "../data/outputs/bis_reer_metrics.csv"
COMBINED_CSV = "../data/outputs/etf_weo_combined_annual.csv"

print("Libraries loaded successfully")

In [ ]:
# Load all data
etf = pd.read_csv(ETF_CSV)
weo = pd.read_csv(WEO_CSV)
metadata = pd.read_csv(METADATA_CSV)
crude = pd.read_csv(CRUDE_IMPACT_CSV)
reer = pd.read_csv(REER_CSV)
combined = pd.read_csv(COMBINED_CSV)

print(f"ETF prices: {etf.shape}")
print(f"WEO GDP: {weo.shape}")
print(f"Metadata: {metadata.shape}")
print(f"Crude Oil Impact: {crude.shape}")
print(f"REER: {reer.shape}")
print(f"Combined: {combined.shape}")

In [ ]:
# Region mapping
COUNTRY_TO_REGION = {
    "Australia": "Oceania", "Austria": "Europe", "Belgium": "Europe",
    "Brazil": "Latin America", "Bulgaria": "Europe", "Canada": "North America",
    "China": "Asia", "France": "Europe", "Germany": "Europe",
    "Greece": "Europe", "Hong Kong": "Asia", "India": "Asia",
    "Indonesia": "Asia", "Italy": "Europe", "Japan": "Asia",
    "Kuwait": "Middle East", "Malaysia": "Asia", "Mexico": "Latin America",
    "Netherlands": "Europe", "Pakistan": "Asia", "Philippines": "Asia",
    "Poland": "Europe", "Saudi Arabia": "Middle East", "Singapore": "Asia",
    "South Africa": "Africa", "South Korea": "Asia", "Spain": "Europe",
    "Sweden": "Europe", "Switzerland": "Europe", "Taiwan": "Asia",
    "Thailand": "Asia", "Turkey": "Europe", "United Kingdom": "Europe",
    "United States": "North America", "Vietnam": "Asia",
}

# Helper: CAGR calculation
def calc_cagr(values, years):
    """Calculate CAGR from a list of values."""
    if len(values) < 2 or years <= 0:
        return None
    start, end = values[0], values[-1]
    if pd.isna(start) or pd.isna(end) or start <= 0 or end <= 0:
        return None
    return ((end / start) ** (1 / years) - 1) * 100

print("Helper functions defined")

In [ ]:
# Build the screener table
def build_screener_table(horizon="5Y"):
    """Build the country screener table with configurable horizon."""
    
    horizon_years = {"1Y": 1, "3Y": 3, "5Y": 5, "10Y": 10}[horizon]
    countries = combined["country_name"].unique()
    rows = []
    
    for country in sorted(countries):
        country_data = combined[combined["country_name"] == country].sort_values("year")
        if country_data.empty:
            continue
        
        # Get ticker info
        ticker = country_data["ticker"].iloc[-1]
        ticker_meta = metadata[metadata["ticker"] == ticker]
        currency = ticker_meta["currency"].iloc[0] if not ticker_meta.empty else "USD"
        exchange = ticker_meta["exchange"].iloc[0] if not ticker_meta.empty else "N/A"
        
        # Get GDP data
        gdp_usd = country_data[country_data["gdp_current_usd"].notna()]
        
        # GDP CAGR (USD)
        gdp_cagr = None
        if len(gdp_usd) >= 2:
            recent = gdp_usd.tail(horizon_years)
            if len(recent) >= 2:
                gdp_cagr = calc_cagr(recent["gdp_current_usd"].values, len(recent) - 1)
        
        # ETF CAGR (USD) - calculate from annual returns
        etf_usd = country_data[country_data["etf_return_usd_pct"].notna()]
        etf_cagr = None
        if len(etf_usd) >= 2:
            recent_etf = etf_usd.tail(horizon_years)
            if len(recent_etf) >= 2:
                returns = recent_etf["etf_return_usd_pct"].values
                cumulative = 1.0
                for r in returns:
                    cumulative *= (1 + r / 100)
                etf_cagr = (cumulative ** (1 / (len(recent_etf) - 1)) - 1) * 100 if len(recent_etf) > 1 else returns[-1]
        
        # Macro Gap
        macro_gap = None
        if gdp_cagr is not None and etf_cagr is not None:
            macro_gap = gdp_cagr - etf_cagr
        
        # Oil Impact
        oil_impact = crude[crude["country_name"] == country]["impact_percent_gdp"]
        oil_impact_val = oil_impact.iloc[0] if not oil_impact.empty else None
        
        # REER
        reer_data = reer[reer["country_name"] == country]
        reer_latest = reer_data["current_reer"].iloc[0] if not reer_data.empty else None
        reer_10y_avg = reer_data["avg_10y_reer"].iloc[0] if not reer_data.empty else None
        reer_vs_10y = (reer_latest - reer_10y_avg) if reer_latest and reer_10y_avg else None
        
        # Projections (2026-28)
        projections = weo[(weo["country_name"] == country) & (weo["indicator"] == "NGDPD") & (weo["year"] >= 2026)]
        proj_cagr = None
        if len(projections) >= 2:
            proj_cagr = calc_cagr(projections["value"].astype(float).values, len(projections) - 1)
        
        # FX CAGR
        fx_cagr = country_data["country_lcu_vs_usd_10y_cagr"].iloc[-1] if "country_lcu_vs_usd_10y_cagr" in country_data.columns else None
        
        # Inflation Diff
        inf_diff = country_data["inflation_cpi_diff_10y_avg"].iloc[-1] if "inflation_cpi_diff_10y_avg" in country_data.columns else None
        
        # Currency Gap
        currency_gap = None
        if fx_cagr is not None and inf_diff is not None:
            currency_gap = fx_cagr + inf_diff
        
        # GDP Real CAGR
        gdp_real = country_data[country_data["gdp_real_growth_pct"].notna()]
        gdp_real_cagr = None
        if len(gdp_real) >= 2:
            recent_real = gdp_real.tail(horizon_years)
            if len(recent_real) > 1:
                gdp_real_cagr = calc_cagr(recent_real["gdp_real_growth_pct"].values + 100, len(recent_real) - 1)
            else:
                gdp_real_cagr = recent_real["gdp_real_growth_pct"].iloc[-1]
        
        # GDP LCU CAGR
        gdp_lcu = country_data[country_data["gdp_current_lcu"].notna()]
        gdp_lcu_cagr = None
        if len(gdp_lcu) >= 2:
            recent_lcu = gdp_lcu.tail(horizon_years)
            if len(recent_lcu) >= 2:
                gdp_lcu_cagr = calc_cagr(recent_lcu["gdp_current_lcu"].values, len(recent_lcu) - 1)
        
        # nGDP (USD) 2025
        ngdp_2025 = weo[(weo["country_name"] == country) & (weo["year"] == 2025) & (weo["indicator"] == "NGDPD")]
        ngdp_2025_val = ngdp_2025["value"].astype(float).iloc[0] if not ngdp_2025.empty else None
        
        rows.append({
            "Country": country,
            "Ticker": ticker,
            "nGDP (USD) 2025": ngdp_2025_val,
            "GDP CAGR (USD) %": gdp_cagr,
            "ETF CAGR (USD) %": etf_cagr,
            "Macro Gap %": macro_gap,
            "Proj. Growth (26-28) %": proj_cagr,
            "Oil Impact %": oil_impact_val,
            "REER Index": reer_latest,
            "REER vs 10Y": reer_vs_10y,
            "FX CAGR %": fx_cagr,
            "Inf. Diff CAGR %": inf_diff,
            "Currency Gap %": currency_gap,
            "GDP Real CAGR %": gdp_real_cagr,
            "GDP LCU CAGR %": gdp_lcu_cagr,
            "Region": COUNTRY_TO_REGION.get(country, "Unknown"),
            "Exchange": exchange,
            "Currency": currency,
        })
    
    return pd.DataFrame(rows)

print("build_screener_table() function defined")

In [ ]:
# 🎯 CONFIGURATION CELL - Edit these values to prototype

# Select horizon (1Y, 3Y, 5Y, 10Y)
HORIZON = "5Y"

# Filter by region (set to None to show all)
# Options: "Asia", "Europe", "North America", "Latin America", "Middle East", "Africa", "Oceania"
REGION_FILTER = None  # Change to ["Europe", "Asia"] to filter

# Columns to display (reorder or remove columns here)
MAIN_COLUMNS = [
    "Country",
    "Ticker",
    "nGDP (USD) 2025",
    "GDP CAGR (USD) %",
    "ETF CAGR (USD) %",
    "Macro Gap %",
    "Proj. Growth (26-28) %",
    "Oil Impact %",
    "REER Index",
    "REER vs 10Y",
]

REFERENCE_COLUMNS = [
    "FX CAGR %",
    "Inf. Diff CAGR %",
    "Currency Gap %",
    "GDP Real CAGR %",
    "GDP LCU CAGR %",
    "Region",
    "Exchange",
    "Currency",
]

# Build the table
df = build_screener_table(horizon=HORIZON)

# Apply region filter
if REGION_FILTER:
    df = df[df["Region"].isin(REGION_FILTER)]

# Select columns
all_columns = MAIN_COLUMNS + REFERENCE_COLUMNS
df = df[[c for c in all_columns if c in df.columns]]

# Create editable copy
edited_df = df.copy()

print(f"Table built with {len(edited_df)} countries and {len(edited_df.columns)} columns")
print(f"Horizon: {HORIZON}, Region Filter: {REGION_FILTER}")

In [ ]:
# 📊 DISPLAY TABLE - Re-run this cell after editing edited_df

def style_dashboard(df):
    """Apply conditional formatting similar to Excel dashboard."""
    
    # Green/Red scale for percentage columns
    pct_columns = [
        "GDP CAGR (USD) %", "ETF CAGR (USD) %", "Macro Gap %",
        "Proj. Growth (26-28) %", "FX CAGR %", "Inf. Diff CAGR %",
        "Currency Gap %", "GDP Real CAGR %", "GDP LCU CAGR %", "REER vs 10Y"
    ]
    
    def color_negative_red_positive_green(val):
        """Color values based on positive/negative."""
        if pd.isna(val):
            return ""
        if val >= 10:
            color = "#006600"  # Dark green
        elif val >= 5:
            color = "#009900"  # Medium green
        elif val >= 0:
            color = "#99CC99"  # Light green
        elif val >= -5:
            color = "#FF9999"  # Light red
        elif val >= -10:
            color = "#CC0000"  # Medium red
        else:
            color = "#660000"  # Dark red
        return f"background-color: {color}; color: white" if val < -5 or val > 5 else f"background-color: {color}"
    
    # REER Index (100 = neutral)
    def color_reer(val):
        if pd.isna(val):
            return ""
        if val == 100:
            return ""
        elif val > 110:
            return "background-color: #006600; color: white"
        elif val > 100:
            return "background-color: #99CC99"
        elif val < 90:
            return "background-color: #660000; color: white"
        else:
            return "background-color: #FF9999"
    
    # Oil Impact (lower = better, highlight if UN fallback)
    def color_oil_impact(val):
        if pd.isna(val):
            return ""
        # Yellow highlight for UN fallback countries (could add logic here)
        return ""
    
    styled = df.style
    
    # Apply formatting to percentage columns
    for col in pct_columns:
        if col in df.columns:
            styled = styled.map(color_negative_red_positive_green, subset=[col])
    
    # Apply REER formatting
    if "REER Index" in df.columns:
        styled = styled.map(color_reer, subset=["REER Index"])
    
    # Format numbers
    formatter = {
        "nGDP (USD) 2025": "${:,.0f}M",
        "GDP CAGR (USD) %": "{:.2f}%",
        "ETF CAGR (USD) %": "{:.2f}%",
        "Macro Gap %": "{:.2f}%",
        "Proj. Growth (26-28) %": "{:.2f}%",
        "Oil Impact %": "{:.2f}%",
        "REER Index": "{:.1f}",
        "REER vs 10Y": "{:.1f}",
        "FX CAGR %": "{:.2f}%",
        "Inf. Diff CAGR %": "{:.2f}%",
        "Currency Gap %": "{:.2f}%",
        "GDP Real CAGR %": "{:.2f}%",
        "GDP LCU CAGR %": "{:.2f}%",
    }
    
    for col, fmt in formatter.items():
        if col in df.columns:
            styled = styled.format({col: fmt})
    
    return styled

# Display the styled table
styled_df = style_dashboard(edited_df)
display(styled_df)

print("\n💡 Tip: Edit edited_df in the next cell and re-run this cell to see changes")

In [ ]:
# ✏️ EDIT DATA HERE - Make changes to test formatting/column values
# Example edits (modify and re-run the display cell above):

# Example 1: Manually override a value
# edited_df.loc[edited_df["Country"] == "Germany", "GDP CAGR (USD) %"] = 5.5

# Example 2: Add a new test column
# edited_df["Test Column"] = edited_df["GDP CAGR (USD) %"] * 2

# Example 3: Bulk edit a column
# edited_df["Macro Gap %"] = edited_df["GDP CAGR (USD) %"] - edited_df["ETF CAGR (USD) %"]

# Make your edits below:
# --- YOUR EDITS HERE ---


# --- END YOUR EDITS ---

print("Edits applied. Re-run the DISPLAY TABLE cell to see changes.")
print(f"Current shape: {edited_df.shape}")
print(f"Columns: {list(edited_df.columns)}")

In [ ]:
# 💾 EXPORT OPTIONS

# Export to CSV
edited_df.to_csv("../data/outputs/dashboard_prototype.csv", index=False)
print("Exported to: data/outputs/dashboard_prototype.csv")

# Export to Excel (with basic formatting)
def export_to_excel(df, output_path):
    """Export to Excel with basic formatting."""
    with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
        df.to_excel(writer, sheet_name='Dashboard', index=False)
        
        worksheet = writer.sheets['Dashboard']
        
        # Auto-fit column widths
        for column in worksheet.columns:
            max_length = 0
            column_letter = column[0].column_letter
            for cell in column:
                try:
                    if len(str(cell.value)) > max_length:
                        max_length = len(str(cell.value))
                except:
                    pass
            adjusted_width = (max_length + 2) * 1.2
            worksheet.column_dimensions[column_letter].width = min(adjusted_width, 50)
    
    print(f"Exported to: {output_path}")

export_to_excel(edited_df, "../data/outputs/dashboard_prototype.xlsx")

In [ ]:
# 📈 QUICK VALIDATION - Check data quality

print("=== Data Quality Check ===")
print(f"\nTotal countries: {len(edited_df)}")
print(f"\nMissing values per column:")
print(edited_df.isna().sum())

print("\n=== Summary Statistics ===")
print("\nGDP CAGR (USD) %:")
print(edited_df["GDP CAGR (USD) %"].describe())

print("\nETF CAGR (USD) %:")
print(edited_df["ETF CAGR (USD) %"].describe())

print("\nMacro Gap %:")
print(edited_df["Macro Gap %"].describe())

## 📝 Notes

### Workflow
1. Edit configuration in the **CONFIGURATION CELL** (horizon, columns, filters)
2. Make data edits in the **EDIT DATA HERE** cell
3. Re-run the **DISPLAY TABLE** cell to see changes with conditional formatting
4. Use **EXPORT OPTIONS** to save your prototype

### Column Definitions

**Main Table:**
- **GDP CAGR (USD) %**: Annualized growth of country's economy in USD
- **ETF CAGR (USD) %**: Annualized ETF price return in USD (includes currency effects)
- **Macro Gap %**: GDP CAGR minus ETF CAGR (positive = economy outperformed)
- **Oil Impact %**: Economic sensitivity to $10/barrel oil price change (% of GDP)
- **REER Index**: Real Effective Exchange Rate level (100 = 10Y average)
- **REER vs 10Y**: Deviation from 10-year REER average

**Reference Section:**
- **FX CAGR %**: Annualized currency movement vs USD
- **Inf. Diff CAGR %**: Country inflation CAGR minus USA inflation CAGR
- **Currency Gap %**: FX CAGR + Inflation Differential (PPP alignment)
- **GDP Real/LCU CAGR %**: Additional GDP growth metrics

### Conditional Formatting Rules
- **Green scale**: Positive values (darker = higher)
- **Red scale**: Negative values (darker = more negative)
- **REER Index**: 100 = neutral (no color), >100 = green (overvalued), <100 = red (undervalued)